In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore")

In [2]:
df_train = pd.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s5e5/test.csv")

# Data Preprocessing

In [3]:
num_vars = df_train.drop(columns=['id','Calories']).select_dtypes(include=['int64', 'float64']).columns
cat_vars = ['Sex']

In [4]:
df_train['Sex'] = df_train['Sex'].astype('category')
df_test['Sex'] = df_test['Sex'].astype('category')

# Feature Engineering

In [5]:
from sklearn.preprocessing import StandardScaler

def make_features(df, test=False):
    df_temp = df.copy()

    df_temp.drop(columns=['id'], inplace=True)
    # dummie encoding
    df_temp = pd.get_dummies(df_temp, columns=['Sex'])

    # new features, to be culled off in feature selection
    df_temp['BMI'] = df_temp['Weight'] / (df_temp['Height']/100)**2
    # df_temp['Duration_Heart_Rate'] = df_temp['Duration'] * df_temp['Heart_Rate']
    # df_temp['Duration_Body_Temp'] = df_temp['Duration'] * df_temp['Body_Temp']
    # df_temp['Heart_Rate_Body_Temp'] = df_temp['Heart_Rate'] * df_temp['Body_Temp']
    # df_temp['Weight_per_Age'] = df_temp['Weight'] / (df_temp['Age'] + 1)
    # df_temp['Temp_per_Heart'] = df_temp['Body_Temp'] / (df_temp['Heart_Rate'] + 1)
    # df_temp['Height_per_Age'] = df_temp['Height'] / (df_temp['Age'] + 1)
    # df_temp['Heart_Duration_Ratio'] = df_temp['Heart_Rate'] / (df_temp['Duration'] + 1)
    # df_temp['Duration_Squared'] = df_temp['Duration'] ** 2
    # df_temp['Body_Temp_Squared'] = df_temp['Body_Temp'] ** 2
    
    # standardization
    # if test == True:
    #     features_to_scale = df_temp.select_dtypes(include=['int64', 'float64']).columns
    # else:
    #     features_to_scale = df_temp.drop(columns=['Calories']).select_dtypes(include=['int64', 'float64']).columns
    # scaler = StandardScaler()
    # df_temp[features_to_scale] = scaler.fit_transform(df_temp[features_to_scale])
    
    return df_temp

# for predicting log of outcome rather than just outcome
def make_features_log(df):
    df_temp = df.copy()

    df_temp = make_features(df_temp)

    df_temp['log_calories'] = np.log1p(df_temp['Calories'])
    
    return df_temp

df_train1 = make_features(df_train)
df_train2 = make_features_log(df_train)

# Model

## XGB Baseline

In [6]:
SEED = 30

In [7]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split

In [8]:
# without the fancy features
X = df_train.drop(columns=['id', 'Calories'])
X = pd.get_dummies(X, columns=['Sex'])
y = df_train['Calories']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=SEED)

# with the fancy features
X1 = df_train1.drop(columns=['Calories'])
y1 = df_train1['Calories']
X_train1, X_val1, _, _ = train_test_split(X1, y1, test_size=0.3, random_state=SEED)

# with log calories
X2 = df_train2.drop(columns=['Calories', 'log_calories'])
y2 = df_train2['log_calories']
X_train2, X_val2, y_train2, y_val2 = train_test_split(X2, y2, test_size=0.3, random_state=SEED)

In [9]:
# # baseline without new features
# xgb_baseline = xgb.XGBRegressor(enable_calegorical=True)
# xgb_baseline.fit(X_train, y_train)

# y_val_pred = xgb_baseline.predict(X_val)
# score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
# print(f'XGB Baseline Score: {score}')

In [10]:
# # baseline with new features
# xgb_baseline1 = xgb.XGBRegressor(enable_calegorical=True)
# xgb_baseline1.fit(X_train1, y_train)

# y_val_pred = xgb_baseline1.predict(X_val1)
# y_val_pred = np.maximum(y_val_pred, 0)
# score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
# print(f'XGB Baseline Score With new Features: {score}')

In [11]:
# # baseline with log calories
# xgb_baseline2 = xgb.XGBRegressor(enable_calegorical=True)
# xgb_baseline2.fit(X_train2, y_train2)

# y_val_pred = xgb_baseline2.predict(X_val2)
# y_val_pred = np.expm1(y_val_pred)
# y_val_pred = np.maximum(y_val_pred, 0)
# score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
# print(f'XGB Baseline Score With log Calories: {score}')

## Big Tuna

In [12]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [13]:
def rmsle_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    y_pred = np.maximum(y_pred, 0)
    loss = np.sqrt(mean_squared_log_error(y_true, y_pred))
    return 'RMSLE', loss

# Function to run k-fold cross-validation with XGBoost and MSLE
def xgb_cv_rmsle(X, y, params, num_folds=5, debug=False, log=False, y_act=y):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        y_val_act = y_act.iloc[val_idx]
        
        model = xgb.XGBRegressor(
            **params
        )
        model.fit(X_train,y_train)
        
        y_val_pred = model.predict(X_val)
        y_val_pred = np.maximum(0, y_val_pred)

        if log == True:
            y_val_pred = np.expm1(y_val_pred)
            
        score = np.sqrt(mean_squared_log_error(y_val_act, y_val_pred))

        if debug == True:
            print(score)
            
        fold_scores.append(score)
        
    return fold_scores

In [14]:
def objective(trial):
    params = {
        # "objective": "reg:squarederror",
        "eval_metric" : "rmse",
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, step=0.01),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
        "max_bin": trial.suggest_int("max_bin", 256, 2048),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 0.1, step=0.01),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "n_estimators": trial.suggest_int("n_estimators", 50, 2000),
        "max_delta_step": trial.suggest_int("max_delta_step", 1, 10),
        "random_state": SEED
    }

    score = np.mean(xgb_cv_rmsle(X=X2, y=y2, params=params, debug=False, log=True))
    return score

In [15]:
# %%time
# study = optuna.create_study(direction='minimize',
#                             sampler = optuna.samplers.RandomSampler(seed=SEED),
#                             study_name = "BIG BLUE FIN TUNA!!")
# study.optimize(objective, n_trials=100, show_progress_bar=True, )

In [16]:
# best_params = study.best_params
# print(f'Best Trial Params: {best_params}')

# print(f'Best Trial Value: {study.best_trial.value}')

In [17]:
# # for saving versions
best_params = {'learning_rate': 0.03, 'max_depth': 14, 'subsample': 1.0, 'colsample_bytree': 0.7, 'max_bin': 1774, 'min_child_weight': 4, 'gamma': 0.02, 'lambda': 0.5720902445525685, 'alpha': 4.286145035324733, 'grow_policy': 'lossguide', 'n_estimators': 546, 'max_delta_step': 2}

# Submission

In [18]:
best_model = xgb.XGBRegressor(**best_params)
best_model.fit(X_train2, y_train2)

XGBRegressor(alpha=4.286145035324733, base_score=None, booster=None,
             callbacks=None, colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.02, grow_policy='lossguide', importance_type=None,
             interaction_constraints=None, lambda=0.5720902445525685,
             learning_rate=0.03, max_bin=1774, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=2, max_depth=14,
             max_leaves=None, min_child_weight=4, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=546,
             n_jobs=None, ...)

In [19]:
y_val_pred = best_model.predict(X_val2)
y_val_pred = np.expm1(y_val_pred)
score = mean_squared_log_error(y_val_pred, y_val)
print(score)

0.003654624154379966


In [20]:
X_train2.head()

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Sex_female,Sex_male,BMI
365047,26,166.0,69.0,21.0,98.0,40.3,True,False,25.039919
309996,21,192.0,102.0,25.0,101.0,40.5,False,True,27.669271
532535,21,161.0,60.0,19.0,96.0,40.7,True,False,23.147255
223844,47,194.0,99.0,17.0,99.0,40.4,False,True,26.304602
190542,75,193.0,102.0,1.0,87.0,37.7,False,True,27.383285


In [21]:
df_test1 = make_features(df_test, test=True)
df_test1.head()

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Sex_female,Sex_male,BMI
0,45,177.0,81.0,7.0,87.0,39.8,False,True,25.854639
1,26,200.0,97.0,20.0,101.0,40.5,False,True,24.250000
2,29,188.0,85.0,16.0,102.0,40.4,True,False,24.049344
3,39,172.0,73.0,20.0,107.0,40.6,True,False,24.675500
4,30,173.0,67.0,16.0,94.0,40.5,True,False,22.386314


In [22]:
y_test_pred = best_model.predict(df_test1)
y_test_pred = np.expm1(y_test_pred)

submission = pd.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")
submission['Calories'] = y_test_pred
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Calories
0,750000,27.590202
1,750001,107.296532
2,750002,86.138039
3,750003,126.423424
4,750004,76.693069


# TESTING SHIT

#